# GMU — Monotone-Projects Nash Equilibrium

This notebook demonstrates the main algorithm (`find_global_NE_monotone_projects`) and the
step-by-step visualization.

## Visualization for the Monotone-projects algorithm

In [1]:
### -------------------------- Technical part --------------------------
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML

from ne_solvers import find_global_NE_monotone_projects
from visualization import animate_outer_history, animate_outer_history_rates
### -------------------------- Technical part --------------------------


### ------------------------------ INPUTS ------------------------------

a = np.array([10, 9, 8, 3, 2], dtype=float)   # project parameters (nonincreasing)
r_total = np.array([15, 14, 2, 1], dtype=float) # player resources

### ------------------------------ INPUTS ------------------------------


### ------------------------------ ENGINE ------------------------------

solver, marg, k, bc_penalty, d_rows, X, outer_hist = find_global_NE_monotone_projects(
    a,
    r_total,
    max_iter_inner=500,
    tol_row_inner=1e-6,
    bc_tol=1e-8,
    verbose=True,
    debug=False,                   # set True for per-step BC diagnostics
    store_X_in_history=True,       # needed for visualization
)

print(f"Cutoff vector k : {k}")
print(f"BC penalty      : {bc_penalty:.3e}")
print(f"Row disruption  : {d_rows:.3e}")
print()
print("Allocation X (rows = players, cols = projects):")
print(np.round(X, 4))
print()
print("Player marginal rates c_rows:")
print(np.round(marg.c_rows, 6))

ani = animate_outer_history(
    outer_hist,
    a=a,
    r_players=r_total,
    interval_ms=900,
    figsize=(14, 6.5),
)
HTML(ani.data)  # display inline

[init] k=[1, 1, 1, 1]  bc=8.835e+00  d_rows=0.000e+00
[step 1] expand_new j=0 -> k[j]=2  bc=7.767e+00
[step 1] expand_new j=1 -> k[j]=2  bc=7.689e+00
[step 1] expand_new j=2 -> k[j]=2  bc=7.688e+00
[step 1] expand_new j=3 -> k[j]=2  bc=7.688e+00
[step 2] expand_new j=0 -> k[j]=3  bc=2.611e+00
[step 2] expand_new j=1 -> k[j]=3  bc=2.560e+00
[step 2] expand_new j=2 -> k[j]=3  bc=2.559e+00
[step 2] expand_new j=3 -> k[j]=3  bc=2.559e+00
[step 3] expand_new j=0 -> k[j]=4  bc=1.519e+00
[step 3] expand_new j=1 -> k[j]=4  bc=1.514e+00
[step 3] expand_new j=2 -> k[j]=4  bc=1.514e+00
[step 3] refuse_new j=3 at K=4
[step 4] expand_new j=0 -> k[j]=5  bc=4.994e-01
[step 4] expand_new j=1 -> k[j]=5  bc=0.000e+00
[step 4] refuse_new j=2 at K=5
[done] step=4  k=[5, 5, 4, 3]  bc=0.000e+00  d_rows=1.212e-04
Cutoff vector k : [5, 5, 4, 3]
BC penalty      : 0.000e+00
Row disruption  : 1.212e-04

Allocation X (rows = players, cols = projects):
[[4.7371 4.2581 3.7791 1.3713 0.8545]
 [4.4367 3.9861 3.5355 1

## Marginal Rate Matrix

In [ ]:
from single_player import compute_marginal_rate_matrix
from visualization import plot_allocation_and_rates

# Marginal rate: c[j,i] = a_i * (1 + L_i - x_{j,i}) / (1 + L_i)^2
C_mat = compute_marginal_rate_matrix(a, X)

print("Marginal rate matrix  c[j, i]  (rows = players, cols = projects):")
print(np.round(C_mat, 6))

fig = plot_allocation_and_rates(a, X, r_players=r_total)
plt.show()

## C of R and R of C

In [ ]:
from single_player import R_of_C, C_of_R

# ── Zone parameters ────────────────────────────────────────────────────────
a_zone = np.array([10, 9], dtype=float)   # project parameters in the zone
m_zone = 4                                 # number of players active in zone
k_zone = len(a_zone)                       # number of projects in zone

# ── R(C): given a zone marginal rate C, compute total resources R ──────────
C_input = 1.70934087
R_result = R_of_C(C_input, a_zone, m_zone, k_zone)
print(f"R_of_C(C = {C_input})  =  {R_result:.8f}")

# ── C(R): given total zone resources R, compute equilibrium marginal C ─────
R_input = 32.0
C_result = C_of_R(R_input, a_zone, m_zone, k_zone)
print(f"C_of_R(R = {R_input})   =  {C_result:.8f}")

# ── Round-trip check ───────────────────────────────────────────────────────
R_check = R_of_C(C_result, a_zone, m_zone, k_zone)
print(f"\nRound-trip: R_of_C( C_of_R({R_input}) ) = {R_check:.8f}  "
      f"(error = {abs(R_check - R_input):.2e})")